# G1-Extension: Rollout Decay Shape + Channel-Count Check

**Two separate questions, kept separate throughout:**

**Part A — does the attractor-fidelity advantage fade steadily or drop off a cliff
across the rollout?** G1's original result (2.2-8.5x better dimension preservation
for Panda vs Chronos) only compared endpoints (H=336 vs ground truth). This re-scores
correlation dimension at each 128-step chunk boundary (H=128, 256, 336) instead of
only the end, using the SAME trajectories G1 already generated -- no new model calls
needed for this part.

**Part B — does the gap depend on channel count/coupling, or does it exist even at
C=1?** Important correction from re-reading G1's actual source (not from memory):
**every system in G1 was scored single-channel** -- Double Pendulum kept only theta1,
Lorenz only x, Rossler only its first component, Burgers reduced to its first PCA
channel. There is no channel-count variation in the existing G1 data to re-slice.
This part runs ONE new comparison: Lorenz fed to Panda/Chronos as full 3-channel
(x,y,z) context vs the already-established 1-channel result, scoring dimension-error
on the x-component of the forecast in both cases (same estimator, same scoring,
only the model's INPUT changes) -- so if the gap changes, it isolates to the extra
channel information, not to a different metric.

**Estimator-validity note (per this project's estimator-validation rule):** the GP
correlation-dimension estimator was only gated at N=336 in the original G1 notebook.
Part A trusts it at N=128 and N=192 too -- new small-N gates for those lengths run
below, BEFORE Part A's results are used for anything. If either fails, Part A's
128/192 columns should be read as unvalidated, not the 336 column.

**Dependencies:** this notebook is self-contained (re-defines model loading, harness,
estimator -- copied verbatim from `g1_correlation_dimension.ipynb` where unchanged,
modified only where noted). Fill in `BASELINE_CKPT_PATH` / `ABLATION_CKPT_PATH`
below for Part B's retrained-checkpoint targets (Rossler, Burgers, Harmonic); Part A
and Part B's Lorenz/Double-Pendulum rows only need the published checkpoint.


In [1]:
# ============================================================
# CELL 1 -- IMPORTS + MODEL LOADING (verbatim from g1_correlation_dimension.ipynb)
# ============================================================
import os
import json as _json
import numpy as np
import pandas as pd
import torch
from scipy.integrate import solve_ivp
from scipy.spatial.distance import pdist, squareform
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd as _svd
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model_published = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)
chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)
print("Published checkpoints loaded.")

# --- Retrained baseline_100k/ablation_100k (needed for Part A's Rossler/Burgers/Harmonic rows) ---
BASELINE_CKPT_PATH = "C:/Users/user/Downloads/panda-100k-baseline-checkpoint/model.safetensors"
ABLATION_CKPT_PATH = "C:/Users/user/Downloads/panda-100k-ablation-checkpoint/model.safetensors"

from panda.patchtst.patchtst import PatchTSTForPrediction
from panda.utils.train_utils import load_patchtst_model

def _build_model_config(use_dynamics_embedding):
    return dict(
        mode='predict', context_length=512, prediction_length=128, patch_length=16,
        patch_stride=16, num_hidden_layers=8, d_model=512, num_attention_heads=8,
        channel_attention=True, ffn_dim=512, norm_type='rmsnorm', norm_eps=1e-5,
        attention_dropout=0.0, positional_dropout=0.0, path_dropout=0.0, ff_dropout=0.0,
        bias=True, activation_function='gelu', pre_norm=True, use_cls_token=False,
        init_std=0.02, scaling='std', pooling_type='max', head_dropout=0.0,
        channel_rope=False, max_wavelength=500, rope_percent=0.75, loss='mse',
        distribution_output=None, use_dynamics_embedding=use_dynamics_embedding,
        num_poly_feats=120, poly_degrees=2, rff_trainable=False, rff_scale=1.0,
        num_rff=256, do_mask_input=None, mask_type='random', random_mask_ratio=0.5,
        channel_consistent_masking=False, mask_value=0, num_forecast_mask_patches=3,
        unmasked_channel_indices=None, num_parallel_samples=100,
    )

def load_retrained_checkpoint(weights_path, expected_use_dynamics_embedding, arm_name):
    ckpt_dir = os.path.dirname(weights_path)
    info_path = os.path.join(ckpt_dir, "training_info.json")
    if os.path.exists(info_path):
        with open(info_path) as f:
            info = _json.load(f)
        actual = info.get("use_dynamics_embedding")
        assert actual == expected_use_dynamics_embedding, (
            f"ARM MISMATCH at {weights_path}: training_info.json says "
            f"use_dynamics_embedding={actual}, expected {expected_use_dynamics_embedding}. "
            "Wrong checkpoint attached -- stopping before use."
        )
        print(f"Checkpoint identity verified: run_name={info.get('run_name')}, "
              f"use_dynamics_embedding={actual}")
    else:
        print(f"[WARNING] training_info.json missing at {info_path} -- arm identity "
              f"NOT auto-verified. Confirm manually before trusting {arm_name} results.")

    model_config = _build_model_config(expected_use_dynamics_embedding)
    model = load_patchtst_model(
        mode='predict', model_config=model_config,
        pretrained_encoder_path=None, pretained_checkpoint=None,
    )
    assert os.path.exists(weights_path), f"Weights file not found: {weights_path}"
    if weights_path.endswith(".safetensors"):
        from safetensors.torch import load_file as _load_sf
        state = _load_sf(weights_path)
    elif weights_path.endswith(".bin"):
        state = torch.load(weights_path, map_location='cpu')
    else:
        raise ValueError(f"Unrecognized weights file extension: {weights_path}")
    model.load_state_dict(state, strict=True)
    model = model.to(device).eval()
    pipeline = PatchTSTPipeline(mode='predict', model=model)
    print(f"Loaded and wrapped {arm_name}: {weights_path}")
    return pipeline

baseline_100k = None
ablation_100k = None
if BASELINE_CKPT_PATH and ABLATION_CKPT_PATH:
    try:
        baseline_100k = load_retrained_checkpoint(BASELINE_CKPT_PATH, expected_use_dynamics_embedding=True, arm_name="baseline_100k")
        ablation_100k = load_retrained_checkpoint(ABLATION_CKPT_PATH, expected_use_dynamics_embedding=False, arm_name="ablation_100k")
    except Exception as e:
        print(f"[WARNING] Retrained checkpoint loading failed: {e}")
        print("Rossler/Burgers/Harmonic rows will be skipped. Lorenz/Double-Pendulum rows unaffected.")
else:
    print("[INFO] Checkpoint paths not set -- Rossler/Burgers/Harmonic rows will be skipped "
          "until BASELINE_CKPT_PATH/ABLATION_CKPT_PATH are filled in above.")


Device: cpu
Published checkpoints loaded.
Checkpoint identity verified: run_name=baseline, use_dynamics_embedding=True
Loaded and wrapped baseline_100k: C:/Users/user/Downloads/panda-100k-baseline-checkpoint/model.safetensors
Checkpoint identity verified: run_name=koopman_ablation, use_dynamics_embedding=False
Loaded and wrapped ablation_100k: C:/Users/user/Downloads/panda-100k-ablation-checkpoint/model.safetensors


In [2]:
# ============================================================
# CELL 2 -- ROLLOUT HARNESS
# MODIFIED from g1_correlation_dimension.ipynb's Cell 2: panda_forecast_traj now
# also returns the cumulative prediction at each internal chunk boundary (needed
# for Part A's fraction-of-horizon scoring), not just the final concatenated
# trajectory. The forecasting logic itself (chunk size, re-conditioning) is
# UNCHANGED -- this only adds bookkeeping of intermediate state, it does not
# change what gets predicted or how.
# ============================================================
CONTEXT_LEN = 512
TRAIN_H = 128

def instance_norm_window(x_CT):
    mu = x_CT.mean(axis=1, keepdims=True)
    std_raw = x_CT.std(axis=1, keepdims=True)
    std = np.where(std_raw < 1e-6, 1.0, std_raw)
    return (x_CT - mu) / std, mu, std

def panda_forecast_traj_checkpointed(model, context_np, horizon):
    """Same autoregressive rollout as the original panda_forecast_traj, but also
    returns a dict {cumulative_steps: cumulative_prediction_array} at each chunk
    boundary, so Part A can score correlation dimension at H=128, 256, 336
    without re-running inference three times."""
    remaining = horizon
    ctx = context_np.copy()
    preds = []
    checkpoints = {}
    done = 0
    while remaining > 0:
        h = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = model.predict(context_t, h, limit_prediction_length=False, sliding_context=True)
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
        done += h
        checkpoints[done] = np.concatenate(preds, axis=1).copy()
    return checkpoints[horizon], checkpoints

def chronos_forecast_traj_checkpointed(context_np, horizon):
    """Chronos has no internal chunking (single call up to its own length limits),
    so there are no natural chunk boundaries to checkpoint. To compare against
    Panda's chunk boundaries fairly, this calls Chronos separately at each target
    horizon (128, 256, 336) -- each an independent single forecast, NOT a
    truncation of the H=336 call, since Chronos's forecast for a given horizon
    is not guaranteed to be a prefix of its forecast for a longer horizon."""
    checkpoints = {}
    for h_target in [128, 256, horizon]:
        if h_target > horizon:
            continue
        ctx = torch.tensor(context_np, dtype=torch.float32)
        with torch.no_grad():
            out = chronos_model.predict(ctx, prediction_length=h_target, num_samples=1)
        checkpoints[h_target] = out[:, 0, :].cpu().numpy()
    return checkpoints[horizon], checkpoints

print("Checkpointed rollout harness defined.")


Checkpointed rollout harness defined.


In [3]:
# ============================================================
# CELL 3 -- GP CORRELATION DIMENSION ESTIMATOR (verbatim from g1_correlation_dimension.ipynb)
# ============================================================
def _mutual_information(x, y, n_bins=16):
    c_xy, _, _ = np.histogram2d(x, y, bins=n_bins)
    p_xy = c_xy / c_xy.sum()
    p_x = p_xy.sum(axis=1)
    p_y = p_xy.sum(axis=0)
    p_x_p_y = np.outer(p_x, p_y)
    nz = p_xy > 0
    return float(np.sum(p_xy[nz] * np.log(p_xy[nz] / p_x_p_y[nz])))

def _tau_from_autocorr(x_1d, max_tau):
    x = x_1d - x_1d.mean()
    denom = np.dot(x, x)
    autocorr = np.array([np.dot(x[:-t], x[t:]) / denom if t > 0 else 1.0
                          for t in range(1, max_tau + 1)])
    zero_cross = np.where(np.diff(np.sign(autocorr)))[0]
    return int(zero_cross[0]) + 1 if len(zero_cross) > 0 else 1

def select_tau(x_1d, max_tau=50, n_bins=16, smooth_window=3, hold_window=2):
    max_tau = min(max_tau, len(x_1d) // 4)
    if max_tau < smooth_window + hold_window + 2:
        return _tau_from_autocorr(x_1d, max(max_tau, 5))
    mi_vals = np.array([_mutual_information(x_1d[:-t], x_1d[t:], n_bins=n_bins)
                         for t in range(1, max_tau + 1)])
    kernel = np.ones(smooth_window) / smooth_window
    mi_smooth = np.convolve(mi_vals, kernel, mode='valid')
    for i in range(1, len(mi_smooth) - hold_window):
        if mi_smooth[i] < mi_smooth[i - 1] and all(
            mi_smooth[i] <= mi_smooth[i + j] for j in range(1, hold_window + 1)
        ):
            return i + 1
    return _tau_from_autocorr(x_1d, max_tau)

def takens_embed(x_1d, m, tau=1):
    n = len(x_1d) - (m - 1) * tau
    if n <= 0:
        raise ValueError(f"Signal too short for embedding dimension m={m}, tau={tau}")
    return np.array([x_1d[i : i + m * tau : tau] for i in range(n)])

def correlation_dimension(x_1d, m, tau=None, theiler_window=None, n_r=30):
    if tau is None:
        tau = select_tau(x_1d)
    embedded = takens_embed(x_1d, m, tau)
    n_pts = embedded.shape[0]
    if theiler_window is None:
        theiler_window = max(int(0.02 * n_pts), 10)
    dists = squareform(pdist(embedded))
    idx = np.arange(n_pts)
    close_mask = np.abs(idx[:, None] - idx[None, :]) < theiler_window
    dists_masked = dists.copy()
    dists_masked[close_mask] = np.inf
    flat_dists = dists_masked[np.triu_indices(n_pts, k=1)]
    flat_dists = flat_dists[np.isfinite(flat_dists)]
    if len(flat_dists) < 50:
        return np.nan
    d_min, d_max = np.percentile(flat_dists, [1, 99])
    r_values = np.logspace(np.log10(d_min + 1e-12), np.log10(d_max), n_r)
    log_r, log_C = [], []
    for r in r_values:
        c_r = np.mean(flat_dists < r)
        if c_r > 0:
            log_r.append(np.log(r))
            log_C.append(np.log(c_r))
    if len(log_r) < 5:
        return np.nan
    log_r, log_C = np.array(log_r), np.array(log_C)
    lo, hi = np.percentile(log_r, [20, 80])
    mask = (log_r >= lo) & (log_r <= hi)
    if mask.sum() < 3:
        mask = np.ones_like(log_r, dtype=bool)
    slope, _ = np.polyfit(log_r[mask], log_C[mask], deg=1)
    return slope

print("GP estimator defined (verbatim from G1).")


GP estimator defined (verbatim from G1).


## Validation Gates

Reruns the original G1 gates (sine, Lorenz, white noise at N~4000-6000) since this
is a fresh notebook, **plus new small-N gates at N=128 and N=192** -- the two
intermediate lengths Part A needs that the original G1 notebook never validated
(it only checked N=336, matching its own H=336 endpoint). Same signals, same pass
criteria as the original N=336 gate, not relaxed for being shorter.

**If N=128 or N=192 fails while N=336 passes:** Part A's H=336 column can be
trusted; its H=128/H=192 columns should be read as directional only, not as
reliable absolute numbers -- flag this explicitly in whatever you show Toan,
don\'t just report all three columns as equally solid.


In [4]:
# ============================================================
# GATES -- long-N (original G1 thresholds) + new small-N (128, 192, 336)
# ============================================================
def simulate_lorenz_for_gate(n=6000, dt=0.01, sigma=10, rho=28, beta=8/3):
    x, y, z = 0.1, 0.0, 0.0
    xs = [x]
    for _ in range(n - 1):
        dx = sigma*(y-x); dy = x*(rho-z)-y; dz = x*y-beta*z
        x += dt*dx; y += dt*dy; z += dt*dz
        xs.append(x)
    return np.array(xs)

def run_gate_suite(N, label, sine_tol=0.3, lorenz_target=2.05, lorenz_tol=0.5, noise_growth_min=1.5):
    print("="*70)
    print(f"GATE SUITE: N={N} ({label})")
    print("="*70)

    t = np.linspace(0, 200, max(N, 4000))[-N:]
    sine = np.sin(t)
    m_max_a = 5 if N >= 200 else 4
    sine_tau = select_tau(sine)
    sine_sweep = {m: correlation_dimension(sine, m, tau=sine_tau) for m in range(1, m_max_a + 1)}
    a_est = sine_sweep[m_max_a]
    a_pass = (not np.isnan(a_est)) and abs(a_est - 1.0) <= sine_tol
    print(f"  Gate A (sine, d=1): d_hat(m={m_max_a})={a_est:.3f} -> {'PASS' if a_pass else 'FAIL'}")

    lorenz = simulate_lorenz_for_gate(n=max(N + 1000, 6000))[-N:]
    m_max_b = 6 if N >= 200 else 5
    lorenz_tau = select_tau(lorenz)
    lorenz_sweep = {m: correlation_dimension(lorenz, m, tau=lorenz_tau) for m in range(1, m_max_b + 1)}
    b_est = lorenz_sweep[m_max_b]
    b_pass = (not np.isnan(b_est)) and abs(b_est - lorenz_target) <= lorenz_tol
    print(f"  Gate B (Lorenz, d~{lorenz_target}): d_hat(m={m_max_b})={b_est:.3f} -> {'PASS' if b_pass else 'FAIL'}")

    rng = np.random.default_rng(0)
    noise = rng.standard_normal(N)
    m_max_c = 6 if N >= 200 else 5
    noise_tau = select_tau(noise)
    noise_sweep = {m: correlation_dimension(noise, m, tau=noise_tau) for m in range(1, m_max_c + 1)}
    growth = noise_sweep[m_max_c] - noise_sweep[2]
    c_pass = (not np.isnan(growth)) and growth >= noise_growth_min
    print(f"  Gate C (noise, should grow): growth={growth:.3f} -> {'PASS' if c_pass else 'FAIL'}")

    all_pass = a_pass and b_pass and c_pass
    print(f"  N={N} ALL_PASS: {all_pass}")
    print()
    return all_pass

GATES_336 = run_gate_suite(336, "matches original G1 endpoint")
GATES_192 = run_gate_suite(192, "NEW -- needed for Part A intermediate column")
GATES_128 = run_gate_suite(128, "NEW -- needed for Part A intermediate column")

print("="*70)
print(f"Summary: N=128 pass={GATES_128}, N=192 pass={GATES_192}, N=336 pass={GATES_336}")
if not (GATES_128 and GATES_192):
    print("[NOTE] Part A will still run below, but H=128 and/or H=192 columns should be")
    print("read as directional only wherever the corresponding gate failed -- not as")
    print("trustworthy absolute dimension values. H=336 stands on the original G1 validation.")


GATE SUITE: N=336 (matches original G1 endpoint)
  Gate A (sine, d=1): d_hat(m=5)=1.222 -> PASS
  Gate B (Lorenz, d~2.05): d_hat(m=6)=3.846 -> FAIL
  Gate C (noise, should grow): growth=1.609 -> PASS
  N=336 ALL_PASS: False

GATE SUITE: N=192 (NEW -- needed for Part A intermediate column)
  Gate A (sine, d=1): d_hat(m=4)=2.229 -> FAIL
  Gate B (Lorenz, d~2.05): d_hat(m=5)=2.855 -> FAIL
  Gate C (noise, should grow): growth=1.358 -> FAIL
  N=192 ALL_PASS: False

GATE SUITE: N=128 (NEW -- needed for Part A intermediate column)
  Gate A (sine, d=1): d_hat(m=4)=1.884 -> FAIL
  Gate B (Lorenz, d~2.05): d_hat(m=5)=2.607 -> FAIL
  Gate C (noise, should grow): growth=1.195 -> FAIL
  N=128 ALL_PASS: False

Summary: N=128 pass=False, N=192 pass=False, N=336 pass=False
[NOTE] Part A will still run below, but H=128 and/or H=192 columns should be
read as directional only wherever the corresponding gate failed -- not as
trustworthy absolute dimension values. H=336 stands on the original G1 validatio

## Part A: Fraction-of-Horizon Decay Curve

System generators copied verbatim from `g1_correlation_dimension.ipynb` (same
seeds, same physics, same PCA reduction for Burgers) so these are the *same*
trajectories G1 originally scored -- only the scoring resolution changes (three
points along the rollout instead of one).


In [5]:
# ============================================================
# PART A -- system generators (verbatim from g1_correlation_dimension.ipynb)
# ============================================================
def simulate_double_pendulum_corrected(n_steps=4000, dt=0.02, g=9.81, l1=1.0, l2=1.0,
                                         m1=1.0, m2=1.0, seed=42):
    rng = np.random.default_rng(seed)
    th1_0, th2_0 = rng.uniform(-np.pi, np.pi, 2)
    ic = [th1_0, th2_0, 0.0, 0.0]
    def rhs(t, y):
        th1, th2, w1, w2 = y
        delta = th1 - th2
        den = 2*m1 + m2 - m2*np.cos(2*delta)
        a1 = (-g*(2*m1+m2)*np.sin(th1) - m2*g*np.sin(th1-2*th2)
              - 2*np.sin(delta)*m2*(w2**2*l2 + w1**2*l1*np.cos(delta))) / (l1*den)
        a2 = (2*np.sin(delta)*(w1**2*l1*(m1+m2) + g*(m1+m2)*np.cos(th1)
              + w2**2*l2*m2*np.cos(delta))) / (l2*den)
        return [w1, w2, a1, a2]
    sol = solve_ivp(rhs, [0, n_steps*dt], ic,
                     t_eval=np.linspace(0, n_steps*dt, n_steps),
                     method='RK45', rtol=1e-9, atol=1e-10)
    return sol.y[0].astype(np.float32)

def simulate_lorenz_rho(rho, n=5000, dt=0.01):
    x, y, z = 0.1, 0.0, 0.0
    xs = [x]
    for _ in range(n-1):
        dx = 10*(y-x); dy = x*(rho-z)-y; dz = x*y-(8/3)*z
        x += dt*dx; y += dt*dy; z += dt*dz
        xs.append(x)
    return np.array(xs, dtype=np.float32)

def simulate_rossler_for_gate(n_steps=4000, dt=0.05, a=0.2, b=0.2, c=5.7, seed=42):
    rng = np.random.default_rng(seed)
    def rhs(t, y):
        return [-y[1]-y[2], y[0]+a*y[1], b+y[2]*(y[0]-c)]
    ic = rng.standard_normal(3)
    sol = solve_ivp(rhs, [0, n_steps*dt], ic,
                     t_eval=np.linspace(0, n_steps*dt, n_steps),
                     method='RK45', rtol=1e-9, atol=1e-9)
    return sol.y[0].astype(np.float32)

def simulate_burgers_stable(T=1000, N_x=128, nu=1.0, seed=42):
    rng = np.random.default_rng(seed)
    dx = 2 * np.pi / N_x
    dt_diff = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv = 0.4 * dx
    dt = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub = max(1, int(np.ceil(dt_record / dt)))
    dt_act = dt_record / n_sub
    k = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op = -nu * k**2
    u0_hat = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m] += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias
    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin
    U = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1 = rhs_hat(u_hat)
            k2 = rhs_hat(u_hat + 0.5*dt_act*k1)
            k3 = rhs_hat(u_hat + 0.5*dt_act*k2)
            k4 = rhs_hat(u_hat + dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
    return U

def pca_reduction(U, n_components):
    U_c = U - U.mean(axis=0, keepdims=True)
    n_c = min(n_components, min(U_c.shape)-1)
    _, _, Vt = _svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)

def simulate_harmonic_stable(omega=1.0, dt=0.05, n_steps=4000, seed=42):
    rng = np.random.default_rng(seed)
    def rhs(t, y):
        x, v = y
        return [v, -omega**2 * x]
    ic = [float(rng.standard_normal()), float(rng.standard_normal())]
    sol = solve_ivp(rhs, [0, n_steps*dt], ic,
                     t_eval=np.linspace(0, n_steps*dt, n_steps),
                     method='RK45', rtol=1e-8, atol=1e-8)
    return sol.y[0].astype(np.float32)

print("Part A generators defined (verbatim from G1).")


Part A generators defined (verbatim from G1).


In [6]:
# ============================================================
# PART A -- evaluation: score correlation dimension at H=128, 256, 336
# ============================================================
H_CHECKPOINTS = [128, 256, 336]
GATE_STATUS = {128: GATES_128, 192: GATES_192, 256: GATES_128 and GATES_192, 336: GATES_336}
# 256 has no dedicated gate; conservatively requires both 128 and 192 to have passed.

def score_prefix(traj_1d, m=5):
    return correlation_dimension(traj_1d, m=m)

def run_partA_system(name, series_1d, model_published=None, use_retrained=False, H=336):
    data = series_1d[None, :]
    C, T = data.shape
    start = (T - CONTEXT_LEN - H) // 2
    ctx_raw = data[:, start:start+CONTEXT_LEN]
    tgt_raw = data[:, start+CONTEXT_LEN:start+CONTEXT_LEN+H]
    ctx_norm, mu, std = instance_norm_window(ctx_raw)
    tgt_norm = (tgt_raw - mu) / std

    rows = []
    if use_retrained:
        if baseline_100k is None or ablation_100k is None:
            print(f"  [SKIP] {name}: retrained checkpoints not loaded")
            return rows
        _, panda_ckpts = panda_forecast_traj_checkpointed(baseline_100k, ctx_norm, H)
        _, ablation_ckpts = panda_forecast_traj_checkpointed(ablation_100k, ctx_norm, H)
        model_label_a, model_label_b = "baseline_100k", "ablation_100k"
        second_ckpts = ablation_ckpts
    else:
        _, panda_ckpts = panda_forecast_traj_checkpointed(model_published, ctx_norm, H)
        model_label_a, model_label_b = "panda_published", None
        second_ckpts = None
    _, chronos_ckpts = chronos_forecast_traj_checkpointed(ctx_norm, H)

    for h in H_CHECKPOINTS:
        if h > H or h not in panda_ckpts or h not in chronos_ckpts:
            continue
        d_gt = score_prefix(tgt_norm[0, :h])
        d_a = score_prefix(panda_ckpts[h][0])
        d_b = score_prefix(second_ckpts[h][0]) if second_ckpts is not None else np.nan
        d_c = score_prefix(chronos_ckpts[h][0])
        row = {
            "system": name, "h": h, "fraction_of_horizon": round(h / H, 3),
            "gate_status_at_h": GATE_STATUS.get(h, False),
            "d_ground_truth": d_gt,
            f"d_{model_label_a}": d_a,
            "d_chronos": d_c,
            f"err_{model_label_a}": abs(d_a - d_gt) if not np.isnan(d_a) else np.nan,
            "err_chronos": abs(d_c - d_gt) if not np.isnan(d_c) else np.nan,
        }
        if model_label_b:
            row[f"d_{model_label_b}"] = d_b
            row[f"err_{model_label_b}"] = abs(d_b - d_gt) if not np.isnan(d_b) else np.nan
        print(f"  {name} h={h} (frac={row['fraction_of_horizon']}): "
              f"gt={d_gt:.3f} {model_label_a}={d_a:.3f} chronos={d_c:.3f}"
              + (f" {model_label_b}={d_b:.3f}" if model_label_b else ""))
        rows.append(row)
    return rows

partA_rows = []
print("=== Part A: published-checkpoint systems (Double Pendulum, Lorenz rho=10) ===")
partA_rows += run_partA_system("double_pendulum",
    simulate_double_pendulum_corrected()[500:], model_published=panda_model_published)
partA_rows += run_partA_system("lorenz_rho10",
    simulate_lorenz_rho(rho=10)[500:], model_published=panda_model_published)

print()
print("=== Part A: retrained-checkpoint systems (Rossler, Burgers, Harmonic) ===")
partA_rows += run_partA_system("rossler",
    simulate_rossler_for_gate()[500:], use_retrained=True)
_burgers_U = simulate_burgers_stable(T=1000, N_x=128, nu=1.0, seed=42)
_burgers_pca = pca_reduction(_burgers_U, 16)
partA_rows += run_partA_system("burgers_nu1p0", _burgers_pca.T[0], use_retrained=True)
partA_rows += run_partA_system("harmonic",
    simulate_harmonic_stable()[500:], use_retrained=True)

partA_df = pd.DataFrame(partA_rows)
partA_df.to_csv("g1ext_partA_fraction_of_horizon.csv", index=False)
print()
print("Saved g1ext_partA_fraction_of_horizon.csv")
print(partA_df)


=== Part A: published-checkpoint systems (Double Pendulum, Lorenz rho=10) ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  double_pendulum h=128 (frac=0.381): gt=2.262 panda_published=2.206 chronos=2.196
  double_pendulum h=256 (frac=0.762): gt=2.496 panda_published=2.103 chronos=1.918
  double_pendulum h=336 (frac=1.0): gt=2.378 panda_published=1.815 chronos=1.870


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  lorenz_rho10 h=128 (frac=0.381): gt=2.230 panda_published=2.188 chronos=2.415
  lorenz_rho10 h=256 (frac=0.762): gt=1.856 panda_published=2.472 chronos=1.666
  lorenz_rho10 h=336 (frac=1.0): gt=1.935 panda_published=1.898 chronos=1.599

=== Part A: retrained-checkpoint systems (Rossler, Burgers, Harmonic) ===


...

  rossler h=128 (frac=0.381): gt=1.933 baseline_100k=nan chronos=1.963 ablation_100k=2.218
  rossler h=256 (frac=0.762): gt=2.029 baseline_100k=1.994 chronos=1.871 ablation_100k=1.731
  rossler h=336 (frac=1.0): gt=2.671 baseline_100k=1.748 chronos=2.889 ablation_100k=1.852


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  burgers_nu1p0 h=128 (frac=0.381): gt=1.421 baseline_100k=1.732 chronos=2.687 ablation_100k=2.673
  burgers_nu1p0 h=256 (frac=0.762): gt=1.052 baseline_100k=1.386 chronos=1.183 ablation_100k=1.332
  burgers_nu1p0 h=336 (frac=1.0): gt=0.929 baseline_100k=1.271 chronos=1.062 ablation_100k=1.406


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  harmonic h=128 (frac=0.381): gt=2.088 baseline_100k=2.217 chronos=1.988 ablation_100k=2.064
  harmonic h=256 (frac=0.762): gt=1.392 baseline_100k=1.425 chronos=1.635 ablation_100k=1.914
  harmonic h=336 (frac=1.0): gt=1.222 baseline_100k=1.599 chronos=1.256 ablation_100k=1.886

Saved g1ext_partA_fraction_of_horizon.csv
             system    h  fraction_of_horizon  gate_status_at_h  \
0   double_pendulum  128                0.381             False   
1   double_pendulum  256                0.762             False   
2   double_pendulum  336                1.000             False   
3      lorenz_rho10  128                0.381             False   
4      lorenz_rho10  256                0.762             False   
5      lorenz_rho10  336                1.000             False   
6           rossler  128                0.381             False   
7           rossler  256                0.762             False   
8           rossler  336                1.000             False   
9     b

## Part B: Does the Gap Depend on Channel Count?

One new comparison: Lorenz fed as full 3-channel (x, y, z) context instead of the
1-channel (x only) context G1 originally used. Dimension-error is still scored on
the x-component of the forecast only, with the same estimator -- so this isolates
whether *giving the model cross-channel information* changes rollout fidelity,
without changing what's being measured or how.

**Reading the result:** if the 3-channel gap is similar to or smaller than the
existing 1-channel gap, that argues AGAINST channel-coupling being the (sole)
driver of Panda's rollout-structure advantage, since the advantage already exists
with no cross-channel information available. If the 3-channel gap is
substantially larger, that's evidence channel coupling adds to it.

**What this does NOT establish:** Chronos itself is architecturally channel-
independent regardless of how much context it's given (it forecasts each channel
separately even when given all three) -- so this only tests whether *Panda's*
side of the gap changes with channel count, not a full ablation of Chronos's
independence assumption.


In [7]:
# ============================================================
# PART B -- channel-count check: Lorenz 1-channel vs 3-channel context
# ============================================================
def simulate_lorenz_rho_multichannel(rho, n=5000, dt=0.01):
    x, y, z = 0.1, 0.0, 0.0
    xs, ys, zs = [x], [y], [z]
    for _ in range(n-1):
        dx = 10*(y-x); dy = x*(rho-z)-y; dz = x*y-(8/3)*z
        x += dt*dx; y += dt*dy; z += dt*dz
        xs.append(x); ys.append(y); zs.append(z)
    return np.array([xs, ys, zs], dtype=np.float32)  # shape (3, n)

def run_partB_channel_check(H=336):
    lorenz_3ch = simulate_lorenz_rho_multichannel(rho=10)[:, 500:]  # (3, T)
    T = lorenz_3ch.shape[1]
    start = (T - CONTEXT_LEN - H) // 2

    ctx_raw = lorenz_3ch[:, start:start+CONTEXT_LEN]
    tgt_raw = lorenz_3ch[:, start+CONTEXT_LEN:start+CONTEXT_LEN+H]
    ctx_norm, mu, std = instance_norm_window(ctx_raw)
    tgt_norm = (tgt_raw - mu) / std

    panda_pred, _ = panda_forecast_traj_checkpointed(panda_model_published, ctx_norm, H)
    chronos_pred, _ = chronos_forecast_traj_checkpointed(ctx_norm, H)

    # Score dimension on the x-component (channel 0) only -- same estimator,
    # same scoring convention as the 1-channel result already in G1.
    d_gt = correlation_dimension(tgt_norm[0], m=5)
    d_panda_3ch = correlation_dimension(panda_pred[0], m=5)
    d_chronos_3ch = correlation_dimension(chronos_pred[0], m=5)

    err_panda_3ch = abs(d_panda_3ch - d_gt) if not np.isnan(d_panda_3ch) else np.nan
    err_chronos_3ch = abs(d_chronos_3ch - d_gt) if not np.isnan(d_chronos_3ch) else np.nan
    ratio_3ch = err_chronos_3ch / err_panda_3ch if err_panda_3ch not in (0, np.nan) else np.nan

    print(f"Lorenz, 3-channel context: d_gt={d_gt:.3f} d_panda={d_panda_3ch:.3f} "
          f"d_chronos={d_chronos_3ch:.3f}")
    print(f"  err_panda={err_panda_3ch:.3f}  err_chronos={err_chronos_3ch:.3f}  "
          f"ratio(chronos/panda)={ratio_3ch:.2f}x")

    # Compare against Part A's already-computed 1-channel Lorenz result at H=336
    lorenz_1ch_row = partA_df[(partA_df["system"] == "lorenz_rho10") & (partA_df["h"] == H)]
    if len(lorenz_1ch_row) == 1:
        err_panda_1ch = lorenz_1ch_row["err_panda_published"].values[0]
        err_chronos_1ch = lorenz_1ch_row["err_chronos"].values[0]
        ratio_1ch = err_chronos_1ch / err_panda_1ch if err_panda_1ch not in (0, np.nan) else np.nan
        print()
        print(f"Comparison to Part A's 1-channel Lorenz result (H={H}):")
        print(f"  1-channel: err_panda={err_panda_1ch:.3f}  err_chronos={err_chronos_1ch:.3f}  ratio={ratio_1ch:.2f}x")
        print(f"  3-channel: err_panda={err_panda_3ch:.3f}  err_chronos={err_chronos_3ch:.3f}  ratio={ratio_3ch:.2f}x")
        if ratio_3ch > ratio_1ch * 1.2:
            print("  -> Gap notably LARGER with 3 channels: some support for channel-coupling")
            print("     contributing to the advantage.")
        elif ratio_3ch < ratio_1ch * 0.8:
            print("  -> Gap notably SMALLER with 3 channels: argues against channel-coupling")
            print("     being what drives it (advantage shrinks when more info is available,")
            print("     opposite of what the channel-coupling story would predict).")
        else:
            print("  -> Gap roughly UNCHANGED: consistent with channel count/coupling NOT being")
            print("     the primary driver, since the advantage already exists at 1 channel and")
            print("     adding 2 more doesn't move it much either way.")
    else:
        print("[WARNING] Could not find matching 1-channel Lorenz row in partA_df for comparison "
              "-- run Part A's cell before this one.")

    result = {
        "condition": "3_channel_context", "system": "lorenz_rho10", "h": H,
        "d_ground_truth": d_gt, "d_panda": d_panda_3ch, "d_chronos": d_chronos_3ch,
        "err_panda": err_panda_3ch, "err_chronos": err_chronos_3ch, "ratio": ratio_3ch,
    }
    pd.DataFrame([result]).to_csv("g1ext_partB_channel_check.csv", index=False)
    print()
    print("Saved g1ext_partB_channel_check.csv")
    return result

partB_result = run_partB_channel_check()


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Lorenz, 3-channel context: d_gt=1.935 d_panda=1.839 d_chronos=2.609
  err_panda=0.096  err_chronos=0.674  ratio(chronos/panda)=7.00x

Comparison to Part A's 1-channel Lorenz result (H=336):
  1-channel: err_panda=0.037  err_chronos=0.336  ratio=8.99x
  3-channel: err_panda=0.096  err_chronos=0.674  ratio=7.00x
  -> Gap notably SMALLER with 3 channels: argues against channel-coupling
     being what drives it (advantage shrinks when more info is available,
     opposite of what the channel-coupling story would predict).

Saved g1ext_partB_channel_check.csv


## Interpreting the Two Parts Together

**Part A alone tells you the shape of the decay** -- steady, cliff-edge, or flat --
across the rollout, for however many of the five systems have valid gates at each
checkpoint. Read `gate_status_at_h` in the saved CSV before trusting any H=128/256
row; only H=336 carries the original G1 validation automatically.

**Part B alone tells you whether one specific competing explanation (channel
coupling) survives a direct test**, on one system (Lorenz). It is not proof either
way on its own -- one system, one run, same caveat this project has flagged
repeatedly about small evidence bases. Treat it as a first check, not a verdict --
the same way idea #3 asks you to treat G1 itself.

**What this notebook deliberately does NOT do:** build or train anything
flow-matching-related. Both parts are re-analysis plus one new (but cheap, no
retraining) forward pass, exactly the "cheap tests before committing to
architecture" step discussed with Toan.
